# Understanding the fundamentals of tensor networks and ZX calculus

Mathis Hage

In this notebook, we will assess our understanding of tensor networks and ZX calculus by reviewing basic notions, as well as by working out a few playground examples in python.

We make use of [NumPy](https://numpy.org/) for the tensor representations and the [TensorNetwork](https://github.com/google/TensorNetwork) framework to manipulate tensor networks.

In [1]:
import numpy as np
import tensornetwork as tn

## Tensor Networks

This part is largely based on [Hand-waving and Interpretive Dance: An Introductory Course on Tensor Networks](http://arxiv.org/abs/1603.03039), p. 1-18.

### Tensors

#### Definition

Informally (and for our purposes), a tensor is simply a multidimensional array of numbers :
$$T \in \mathbb{C}^{d_1 \times d_2 \times \cdots \times d_N}$$
where access to each element is done via the notation $T_{i_1 i_2 \cdots i_N}$. $N$ is the *rank* of the tensor. There is a distinction between covariant and contravariant indices which will not be detailed here.

In [6]:
# We can define various tensor elements in numpy

# Some matrices (rank-2 tensors) :
pauli_x = np.array([[0, 1], [1, 0]])
pauli_y = np.array([[0, -1j], [1j, 0]])
pauli_z = np.array([[1, 0], [0, -1]])
identity = np.eye(2)

# Some vectors (rank-1 tensors) :
ket_0 = np.array([[1], [0]])
ket_1 = np.array([[0], [1]])

# Rank-3 tensors (here 3x3x3) :
rand_tensor_3d_1 = np.random.rand(3, 3, 3)
rand_tensor_3d_2 = np.random.rand(3, 3, 3)
print("Random rank-3 tensor 1:\n", rand_tensor_3d_1)

Random rank-3 tensor 1:
 [[[0.61310774 0.02907467 0.25212963]
  [0.21266094 0.03845126 0.64781439]
  [0.34798923 0.92582787 0.10724543]]

 [[0.46303193 0.18397439 0.19750233]
  [0.61157585 0.46694133 0.83094751]
  [0.79212967 0.68281383 0.87958378]]

 [[0.68526187 0.86380553 0.99229636]
  [0.33087977 0.17882888 0.36184824]
  [0.76092614 0.94247154 0.19130768]]]


#### Tensor Operations

Let $A, B$ be two tensors. We can define the *tensor product* $A \otimes B$ to be, component-wise :
$$(A \otimes B)_{i_1 \cdots i_{N_A} ; j_1 \cdots j_{N_B}} := A_{i_1 \cdots i_{N_A}} \cdot B_{j_1 \cdots j_{N_B}}$$

Similar to matrices, we can also define the partial trace operations on dimension-matching indices $x, y$ (using the [Einstein summation convention](https://en.wikipedia.org/wiki/Einstein_notation), i.e. the sum over repeated indices is implicit) :
$$\text{Tr}_{x, y} (A) := A_{i_1 \cdots i_{x-1} \: \alpha \: i_{x+1} \cdots i_{y-1} \: \alpha \: i_{y+1} \cdots i_{N_A}}$$

Finally, an important operation for us is *tensor contraction* of two tensors, which can be defined as a tensor product followed by a trace over the indices we wish to contract (one index $x$ from $A$ and another $y$ from $B$, where the dimensions match) :
$$\text{Tr}_{x, y}(A \otimes B)$$
As a side note, we state that one of the index must be covariant while the other must be contravariant, which disallows certain contractions.

### Tensor Network Notation

A tensor network is simply a diagram describing a combination of multiple tensors using tensor operations (and in particular contraction). A node represents a tensor, and a leg sticking out of this node represents an index of the tensor (which can hence be contracted). We talked about how certain indices cannot be contracted together : this is taken into account by orienting the legs in a conventional way. Below are examples taken from [the Tensor Network](https://tensornetwork.org/diagrams/) website :

<img src="res/tensor_diagrams.png" alt="tensor representation" width="700"/>

Then, contraction is represented by joining the legs (same examples source):

<img src="res/sample_contractions.png" alt="sample contractions" width="700"/>

Notice this explicits the fact that after the contraction, we have a new tensor with dimensions specified by the legs that are still unattached.

We can try to play around with contractions in Python. In particular, we can use analogies with matrix product and trace to test our setup.

In [19]:
# Link the nodes together to form a network

X = tn.Node(pauli_x, name="X")
Z = tn.Node(pauli_z, name="Z")

# --X--Z--
edge_xz = tn.connect(X[1], Z[0])

# Perform the contractions
result = tn.contract(edge_xz)

# Check that our intuition is right
print("Contraction of X and Z:\n", result.tensor, "\nIs it equal to X @ Z = -iY ?", np.allclose(result.tensor, -1j * pauli_y))


Contraction of X and Z:
 [[ 0 -1]
 [ 1  0]] 
Is it equal to X @ Z = -iY ? True


In [14]:
# Try trace analogy
X = tn.Node(pauli_x, name="X")

# both legs of X connected to each other
edge_x = tn.connect(X[1], X[0])

result = tn.contract(edge_x)

print("When both legs of X are contracted together, do we get back zero, the trace of X ?", np.equal(result.tensor, 0))

When both legs of X are contracted together, do we get back zero, the trace of X ? True


In [15]:
Rank3_1 = tn.Node(rand_tensor_3d_1, name="Rank3_1")
Rank3_2 = tn.Node(rand_tensor_3d_2, name="Rank3_2")

edge_r3 = tn.connect(Rank3_1[2], Rank3_2[0])
result_r3 = tn.contract(edge_r3)

print("Check that the contraction of two rank-3 tensors along one leg gives a rank-4 tensor: shape =", result_r3.tensor.shape)

Check that the contraction of two rank-3 tensors along one leg gives a rank-4 tensor: shape = (3, 3, 3, 3)


### Bubbling

Given a tensor network, we rapidly notice that the order in which we perform the contractions impacts the computational complexity : as a simple example, take $A, B \in \mathbb{R}^{3 \times 3}, v \in \mathbb{R}^3$. Suppose we want to evaluate the contraction $A_{ij}B_{jl}v_l$. If we start by contracting the two matrices, we will end up doing a total of $3 \cdot 3 \cdot 3 + 3 \cdot 3 = 36$ operations, while if we start by contracting $B$ and $v$, we will end up doing $3 \cdot 3 + 3 \cdot 3 = 18$ operations. There is a factor 2 difference ! 

In general, we call *bubbling* the order in which tensors are contracted. Finding an optimal bubbling is an NP-complete task.

Playground exercise idea : implement an optimizer (naïve) returning the best bubbling for a network.

### MPS (as a side note for now)

Given an initial state :

$$\ket{\psi} := \sum_{j_1 \cdots j_N} C_{j_1 \cdots j_N} \ket{j_1} \otimes \ket{j_2 \cdots j_N}$$

We can iteratively perform Schmidt decompositions starting from the first index. Without going into details, this leads to the rank 3 tensors $A^{1}, \cdots, A^{N}$ specific to this state, and we can then write :
$$\ket{\psi [A^{1}, \cdots A^N ]} = \sum_{i_1 \cdots i_N} \text{Tr}(A^1_{i_1} \cdots A^N_{i_N}) \ket{i_1 \cdots i_N}$$
This can appear useful for states with low entanglement.

## ZX-Calculus

This part is largely based on [ZX-calculus for the working quantum computer scientist](http://arxiv.org/abs/2012.13966), p. 1-34.

ZX-calculus is essentially a graphical way to represent and reason about linear maps between qubits. A ZX-diagram represents qubits flowing through wires, hitting some nodes, which are linear maps acting on them, and at some point coming out. Such a diagram is composed of multiple instances of a specific set of *generators*.

### Generators

The first generators are the *spiders*. These are nodes that can be green or red, and take a parameter $\alpha$. A green node with $n$ input wires (i.e. connected to the left of the node), $m$ output wires (i.e. connected to the right of the node) and parameter $\alpha$, as follows :

<p align="center">
<img src="res/zx_green_general.svg" alt="general green spider" width="300"/>
</p>

corresponds to the linear map :
$$\ket{\overbrace{0 \cdots 0}^m} \bra{\overbrace{0 \cdots 0}^n} + e^{i \alpha} \ket{\overbrace{1 \cdots 1}^m} \bra{\overbrace{1 \cdots 1}^n}.$$
If the node is red, it corresponds to the linear map :
$$\ket{\overbrace{+ \cdots +}^m} \bra{\overbrace{+ \cdots +}^n} + e^{i \alpha} \ket{\overbrace{- \cdots -}^m} \bra{\overbrace{- \cdots -}^n}.$$

Here are a few examples :
- A green spider with 0 inputs and 1 output : <img src="res/zx_green_basic.svg" alt="sample contractions" width="100"/> corresponds to the vector $\ket{0} + e^{i \alpha} \ket{1}$ ;
- The same spider but red and parameter $\alpha = 0$ corresponds to $\ket{0}$, while with $\alpha = \pi$ it corresponds to $\ket{1}$.
- A green spider with one input and one output corresponds to the rotation matrix (on the Bloch sphere) by an angle $\alpha$ around the $Z$ axis :
$$R_Z(\alpha) = \begin{pmatrix} 1 & 0 \\ 0 & e^{i \alpha} \end{pmatrix}$$

There are also other generators. First, the SWAP gate (from quantum computing theory), simply represented as crossing wires :
<p align="center">
<img src="res/swap.png" alt="swap" width="400"/>
</p>

One can freely slide spiders along these crossing wires. It is clear from this visual representation that SWAP is self-inverse : try composing two SWAP's side by side, you will notice that both qubits end up in their original lane.

Moreover, two other generators are of fundamental importance, namely the *cup* and the *cap* (respectively) :
<p align="center">
<img src="res/cup_cap.png" alt="swap" width="600"/>
</p>

Notice how they represent the Bell state and the Bell effect.

Here also, a spider can freely move on a cup or a cap.

### Only connectivity matters

Because of the fact that the spiders are symmetric, and using the SWAP, cups and caps generators, we arrive at the point that makes ZX-calculus very interesting : this can be summarized by the sentence *"only connectivity matters"*. In fact, we can bend wires in any way we like, see them either as inputs or outputs of spiders, this doesn't change the overall linear map represented by the ZX-diagram as long as we dont change the order of the inputs and outputs of the whole diagram.

This also implies another thing : a ZX-diagram is essentially a tensor network !

ZX-calculus is universal, meaning it can represent any linear map.

### Rules

When working with ZX-diagrams, and in particular when trying to simplify them, one can make use of a set of rules to do so. These rules are summarized in the figure below :

<p align="center">
<img src="res/zx_rules.png" alt="rules" width="900"/>
</p>

(a white spider corresponds to a green one in our previous notation). The letters denote *spider fusion* ($f$), *hadamard* ($h$, $hh$), *identity* ($id$), *commute* ($\pi$), *copy* ($c$), *bialgebra* ($b$). Note that the square box represent the hadamard unitary, which can be built from spiders as well. We also have a set of "meta-rules" :

- Only connectivity matters ;
- Each rule also holds with the inputs and outputs interchanged ;
- Every rule also holds with the colours (white and grey) interchanged ;
- The rules ($c$) and ($b$) can be combined and generalised to the following rule :

<p align="center">
<img src="res/zx_general_bialg.png" alt="general bialgebra" width="600"/>
</p>

- The rules imply the *Hopf rule* :

<p align="center">
<img src="res/zx_hopf.png" alt="general bialgebra" width="500"/>
</p>

- The rules ($π$) and ($c$) can be combined to give the more generic ($a$ can be 0 or 1):

<p align="center">
<img src="res/zx_general_copy.png" alt="general bialgebra" width="300"/>
</p>